# Scenario 4: Developer Productivity with Claude Code & MCP
## Hands-on Production Scenario Workbench
**Certification Focus**: CCAR-F Exam Scenarios 1–6 | **Standard**: Claude 4.5 & Extended Thinking

---

### 📊 Production SLAs & Non-Negotiables
- Principle of Least Privilege: 3-5 specialized tools per agent
- MCP Resources for static catalogs (schemas, specs) to avoid exploratory calls
- Grep for content search, Glob for path matching, Read for context
- Edit fallback protocol: Unique anchor expansion -> Read + Write fallback

---

### 🏗️ Architecture & Decision Flow
```mermaid
flowchart TD
    DevTask["Developer Code Editing Task"] --> ToolSelect{"Tool Selection"}
    
    ToolSelect -->|"Content Pattern"| Grep["Grep(pattern)"]
    ToolSelect -->|"Filename Pattern"| Glob["Glob(pattern)"]
    ToolSelect -->|"Static Schema"| Res["MCP Resource (Direct URI)"]
    ToolSelect -->|"Targeted Patch"| Edit["Edit(old_str, new_str)"]
    
    Edit --> Status{"Edit Result"}
    Status -->|"Unique Match"| Success["Applied"]
    Status -->|"Multiple Matches / Non-Unique"| Fallback["Expand context anchors<br>or fallback to Read + Write"]
```

---

### 🏆 Golden Architectural Rules Reference
| Rule | Production Implementation & Rationale |
|---|---|
| **Principle of Least Privilege (TS 2.3)** | Limit agent tools to 3-5 focused tools to prevent tool description saturation. |
| **MCP Resources vs Tools (TS 2.4)** | Expose read-only catalogs via direct URIs instead of multi-roundtrip executable tools. |
| **Grep vs Glob (TS 2.5)** | Use Grep to find functions/callers in file contents; use Glob to find files by path. |
| **Edit Fallback Protocol (TS 2.5)** | When Edit fails on duplicate occurrences, expand anchor lines or fallback to Read+Write. |
| **Project vs User Scope (TS 2.4)** | Place shared project MCP servers in .mcp.json; personal tools in user home config. |

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns.
- **Case Facts Preservation**: Immutable transactional entities must be maintained in a dedicated Case Facts block outside conversation summaries.



In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_4_developer_productivity.simulation import *

print(' Scenario Workbench Initialized: Scenario 4: Developer Productivity with Claude Code & MCP')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Test MCP Resource Direct Access & Built-in Tools
files = {
    "src/auth.py": "def login(): pass",
    "src/utils.py": "def helper(): pass",
    "tests/test_auth.py": "def test_login(): pass"
}
toolbox = DeveloperProductivityToolbox(files)

# 1. MCP Resource read
mcp_res = toolbox.read_mcp_resource("resource://specs/openapi.json")
print("MCP Resource:", mcp_res["name"])
assert mcp_res["success"] is True

# 2. Grep search
grep_res = toolbox.grep_symbols("def login")
print("Grep Matches:", grep_res)
assert len(grep_res) == 1
assert grep_res[0]["file"] == "src/auth.py"

# 3. Glob paths
glob_res = toolbox.glob_paths("tests/*.py")
print("Glob Matches:", glob_res)
assert len(glob_res) == 1



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Test Edit Tool Collision and Safe Fallback Protocol
calc_files = {"calc.py": "def a(): return 1\ndef b(): return 1\n"}
tb2 = DeveloperProductivityToolbox(calc_files)

# Attempt ambiguous edit
collision = tb2.apply_edit("calc.py", "return 1", "return 2")
print("Collision Result:", collision["status"])
assert collision["success"] is False
assert collision["status"] == "AMBIGUOUS_COLLISION"

# Fallback with expanded anchor
recovery = tb2.execute_edit_with_fallback(
    "calc.py",
    old_str="return 1",
    new_str="return 2",
    expanded_anchor="def a(): return 1"
)
print("Recovery Result:", recovery["final_status"])
assert recovery["final_status"] == "EXPANDED_ANCHOR_SUCCESS"



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.
